# Optimust Model Lab · 01 — Mapping emotion inside Gemma 3

**What this notebook does**
1. Loads **Gemma 3 1B (instruction-tuned)** and Google DeepMind's **Gemma Scope 2** sparse autoencoder (SAE) for layer 13, a ready-made "feature map" of the model's middle layer.
2. Runs a bilingual (English / Dominican-inflected Spanish) probe set through it: **sadness, grief, anxiety, fear**, plus neutral controls.
3. **Maps** which features fire for each concept and links each one to its Neuronpedia page.
4. **Measures separation:** does the model keep *grief* apart from *sadness*, and *anxiety* apart from *fear*? Does what it learned in English carry over to Spanish?
5. **Steers** a feature up or down and watches the output change.
6. *(Optional)* **LoRA fine-tunes** the model, then re-runs the same measurements to see what changed and what broke.

**Before you run**
- Runtime → Change runtime type → **T4 GPU** (free tier is enough).
- On Hugging Face, open [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it) and **accept the Gemma license**.
- Create a Hugging Face **read token** (Settings → Access Tokens). You'll paste it in step 1.

> The probe set has 20 sentences per concept (10 per language). That's enough to explore, not enough to conclude anything. Treat every result as a lead to check, and have native speakers review and expand the sentences before relying on them.

## 0 · Install

In [ ]:
!pip install -q -U sae-lens transformers accelerate peft scikit-learn pandas

## 1 · Log in to Hugging Face

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2 · Configuration
Gemma 3 1B has 26 layers. Gemma Scope 2 publishes its "featured" residual-stream SAEs at layer 13 for this model. `width_16k` means about 16,000 features; `l0_medium` means about 60 features are active per token. You can swap in `width_65k` or `width_262k` later for a finer-grained map.

In [ ]:
MODEL_ID    = "google/gemma-3-1b-it"
SAE_RELEASE = "gemma-scope-2-1b-it-res"
SAE_ID      = "layer_13_width_16k_l0_medium"
LAYER       = 13
NP_BASE     = "https://www.neuronpedia.org/gemma-3-1b-it/13-gemmascope-2-res-16k"   # + /<feature index>

## 3 · Load Gemma 3 and the SAE

In [ ]:
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from sae_lens import SAE

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(device).eval()

_res = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
sae = _res[0] if isinstance(_res, tuple) else _res
sae.eval()
print("SAE features:", sae.cfg.d_sae, "| model width:", model.config.hidden_size, "| device:", device)
try: print("SAE hook:", sae.cfg.metadata.hook_name)
except Exception: pass

## 4 · Helpers: grab layer-13 activations and encode them into features

In [ ]:
def decoder_layers(m):
    """Find the transformer block list (works for the base model and for LoRA-wrapped models)."""
    n = m.config.num_hidden_layers if hasattr(m, "config") else None
    for name, mod in m.named_modules():
        if name.endswith("layers") and isinstance(mod, torch.nn.ModuleList) and (n is None or len(mod) == n):
            return mod
    raise RuntimeError("Couldn't find decoder layers")

_cache = {}
def _capture(mod, inp, out):
    _cache["h"] = (out[0] if isinstance(out, tuple) else out).detach()

@torch.no_grad()
def layer_acts(texts, m=None):
    m = m or model
    hook = decoder_layers(m)[LAYER].register_forward_hook(_capture)
    try:
        enc = tok(texts, return_tensors="pt", padding=True).to(device)
        m(**enc)
    finally:
        hook.remove()
    return _cache["h"].float(), enc["attention_mask"]

def content_mask(mask):
    """1 for real text tokens; 0 for padding and for each sentence's BOS token (which has a huge, uninformative activation)."""
    keep = mask.clone()
    first = mask.float().argmax(1)                      # position of BOS (works for left or right padding)
    keep[torch.arange(len(first)), first] = 0
    return keep

@torch.no_grad()
def feature_matrix(texts, m=None, batch=16):
    """One row per text: each feature's MAX activation over the text's tokens (skipping the BOS token)."""
    rows = []
    for i in range(0, len(texts), batch):
        h, mask = layer_acts(texts[i:i+batch], m)
        f = sae.encode(h)                                   # (batch, tokens, d_sae)
        f = f * content_mask(mask)[..., None]              # drop BOS / padding
        rows.append(f.max(1).values.cpu())
    return torch.cat(rows).numpy()

@torch.no_grad()
def token_view(text, feat, m=None):
    """Print each token with how strongly one feature fires on it."""
    h, _ = layer_acts([text], m)
    acts = sae.encode(h)[0, :, feat].cpu().numpy()
    ids = tok(text, return_tensors="pt")["input_ids"][0]
    for t, a in list(zip(tok.convert_ids_to_tokens(ids), acts))[1:]:
        print(f"{a:7.2f}  {'█' * int(min(a, 40) / 2):20s} {t}")

## 5 · Sanity check: does the SAE fit this layer?
The SAE should rebuild layer-13 activations well. Look for **variance explained above ~0.7**. If it's much lower, the layer or hook point doesn't match; try `LAYER = 12` or `14`. Run this check again after any fine-tuning to see how far the map has drifted.

In [ ]:
@torch.no_grad()
def sae_fit(texts, m=None):
    h, mask = layer_acts(texts, m)
    x = h[content_mask(mask).bool()]
    xr = sae.decode(sae.encode(x))
    fve = 1 - ((x - xr) ** 2).sum() / ((x - x.mean(0)) ** 2).sum()
    return float(fve)

print("variance explained:", round(sae_fit(["The library opens at nine on Saturdays, and my cousin is learning guitar.",
                                            "Desde que murió mi papá, sigo poniendo su plato en la mesa."]), 3))

## 6 · The probe set (English / Spanish)

In [ ]:
import io
PROBE_CSV = 'concept,lang,text\nsadness,en,I\'ve been feeling down all week and I can\'t really say why.\nsadness,en,It made me sad that nobody remembered my birthday.\nsadness,en,"Rainy days just leave me low, like everything is gray."\nsadness,en,I cried a little after the movie ended.\nsadness,en,He seemed blue after his team lost the game.\nsadness,en,I\'m disappointed I didn\'t get the job.\nsadness,en,"Some days I just feel empty and heavy, and I don\'t know why."\nsadness,en,She was sad when her best friend moved to another state.\nsadness,en,"I miss how things used to be, it makes me a little sad."\nsadness,en,It hurts that my sister and I don\'t talk anymore.\nsadness,es,"Llevo toda la semana desanimado y no sé bien por qué."\nsadness,es,Me dio tristeza que nadie se acordara de mi cumpleaños.\nsadness,es,"Los días de lluvia me dejan bajito, como si todo estuviera gris."\nsadness,es,Lloré un poquito cuando se acabó la película.\nsadness,es,Se veía triste después de que su equipo perdió.\nsadness,es,Estoy decepcionada porque no me dieron el trabajo.\nsadness,es,"Hay días que me siento vacío y pesado, y no sé por qué."\nsadness,es,Se puso triste cuando su mejor amiga se mudó a otro estado.\nsadness,es,"Extraño cómo eran las cosas antes, me da un poco de pena."\nsadness,es,Me duele que mi hermana y yo ya no nos hablemos.\ngrief,en,"Since my father died, I keep setting his plate at dinner."\ngrief,en,It\'s been a year since we buried my grandmother and it still doesn\'t feel real.\ngrief,en,I can\'t bring myself to delete my mom\'s voicemails.\ngrief,en,"At the funeral I felt nothing, and then a week later I couldn\'t stop crying."\ngrief,en,Her son passed away last spring and she still wears his jacket.\ngrief,en,Every holiday reminds me that my brother isn\'t here anymore.\ngrief,en,"After the miscarriage, everyone expected me to just move on."\ngrief,en,I keep dreaming that my wife is still alive and then I wake up.\ngrief,en,We scattered his ashes by the ocean he loved.\ngrief,en,"Losing my best friend so suddenly, I don\'t know who I am now."\ngrief,es,"Desde que murió mi papá, sigo poniendo su plato en la mesa."\ngrief,es,Ya hace un año que enterramos a mi abuela y todavía no me lo creo.\ngrief,es,No me atrevo a borrar los mensajes de voz de mi mamá.\ngrief,es,"En el velorio no sentí nada, y una semana después no podía dejar de llorar."\ngrief,es,Su hijo falleció la primavera pasada y ella todavía se pone su chaqueta.\ngrief,es,Cada Navidad me recuerda que mi hermano ya no está.\ngrief,es,"Después de la pérdida del embarazo, todos esperaban que yo siguiera como si nada."\ngrief,es,Sueño que mi esposa sigue viva y luego me despierto.\ngrief,es,Esparcimos sus cenizas en el mar que tanto quería.\ngrief,es,"Perdí a mi mejor amigo de repente, y ya no sé quién soy."\nanxiety,en,"I keep thinking something bad is going to happen, even though nothing is wrong."\nanxiety,en,My mind won\'t stop racing about all the things I might forget tomorrow.\nanxiety,en,"Before every meeting my chest gets tight, even when it\'s routine."\nanxiety,en,I checked the stove three times and I\'m still worried I left it on.\nanxiety,en,"What if I fail the exam, then lose the scholarship, then everything falls apart?"\nanxiety,en,I can\'t sleep because I\'m replaying every conversation from today.\nanxiety,en,I feel on edge all the time and I can\'t relax.\nanxiety,en,"I haven\'t heard back from her in an hour, and I\'m already imagining the worst."\nanxiety,en,"Money is fine right now, but I can\'t stop worrying about next year."\nanxiety,en,I get restless and jittery whenever I have to wait for news.\nanxiety,es,"Siento que algo malo va a pasar, aunque todo está bien."\nanxiety,es,No paro de darle vueltas a todo lo que se me puede olvidar mañana.\nanxiety,es,"Antes de cada reunión se me aprieta el pecho, aunque sea algo normal."\nanxiety,es,Revisé la estufa tres veces y todavía me preocupa haberla dejado prendida.\nanxiety,es,"¿Y si me cuelgan en el examen, pierdo la beca y todo se viene abajo?"\nanxiety,es,No puedo dormir porque repaso cada conversación del día.\nanxiety,es,Vivo con los nervios de punta y no logro relajarme.\nanxiety,es,"Hace una hora que no me contesta, y ya me estoy imaginando lo peor."\nanxiety,es,"Ahorita el dinero está bien, pero no dejo de preocuparme por el año que viene."\nanxiety,es,Me pongo inquieto y ansioso cada vez que tengo que esperar noticias.\nfear,en,A huge dog lunged at me and I froze.\nfear,en,"I heard someone trying the door handle at 3 a.m., and I couldn\'t breathe."\nfear,en,The car skidded on the ice and I thought we were going to crash.\nfear,en,"When the plane dropped suddenly, I grabbed the armrest in terror."\nfear,en,I saw a snake by my foot and jumped back screaming.\nfear,en,The building started shaking and we ran for the stairs.\nfear,en,A man followed me for three blocks and I was terrified.\nfear,en,The smoke alarm went off and I saw flames in the kitchen.\nfear,en,"I lost my child in the crowd for a minute, and I panicked."\nfear,en,The wave knocked me under and I couldn\'t find the surface.\nfear,es,Un perro enorme se me tiró encima y me quedé paralizado.\nfear,es,"Escuché que alguien trataba de abrir la puerta a las tres de la mañana, y no podía respirar."\nfear,es,El carro patinó en el hielo y pensé que nos íbamos a estrellar.\nfear,es,"Cuando el avión cayó de golpe, me agarré del asiento muerta de miedo."\nfear,es,Vi una culebra al lado de mi pie y salté gritando.\nfear,es,El edificio empezó a temblar y corrimos hacia las escaleras.\nfear,es,Un hombre me siguió tres cuadras y yo estaba aterrada.\nfear,es,Sonó la alarma de humo y vi llamas en la cocina.\nfear,es,"Perdí a mi niño entre la gente por un minuto, y me entró el pánico."\nfear,es,La ola me revolcó y no encontraba la superficie.\nneutral,en,I bought rice and beans at the store this afternoon.\nneutral,en,The meeting was moved to Thursday at ten.\nneutral,en,My cousin is painting the living room light green.\nneutral,en,The bus to Scranton leaves every forty minutes.\nneutral,en,I\'m reorganizing the files on my laptop by year.\nneutral,en,We need to replace the battery in the smoke detector.\nneutral,en,The library opens at nine on Saturdays.\nneutral,en,She is learning to play the guitar on weekends.\nneutral,en,The recipe calls for two cups of flour.\nneutral,en,I watered the plants and took out the recycling.\nneutral,es,Compré arroz y habichuelas en el colmado esta tarde.\nneutral,es,La reunión se movió para el jueves a las diez.\nneutral,es,Mi primo está pintando la sala de verde clarito.\nneutral,es,La guagua para Scranton sale cada cuarenta minutos.\nneutral,es,Estoy organizando los archivos de mi laptop por año.\nneutral,es,Hay que cambiarle la batería al detector de humo.\nneutral,es,La biblioteca abre a las nueve los sábados.\nneutral,es,Ella está aprendiendo a tocar guitarra los fines de semana.\nneutral,es,La receta lleva dos tazas de harina.\nneutral,es,Regué las matas y saqué el reciclaje.\n'
df = pd.read_csv(io.StringIO(PROBE_CSV))
# To use your own set instead: upload a CSV with columns concept,lang,text and run
#   df = pd.read_csv("my_probe_set.csv")
print(df.groupby(["concept", "lang"]).size().unstack())

## 7 · Encode every sentence into features

In [ ]:
F = feature_matrix(df.text.tolist())
print("feature matrix:", F.shape, "| avg active features per sentence:", round(float((F > 0).sum(1).mean()), 1))

## 8 · The map: which features belong to which concept?
**Specificity** = how strongly a feature fires on this concept, minus how strongly it fires on neutral sentences. **Hit rate** = the share of the concept's sentences where it fires at all. Click the link to see what each feature responds to across a huge text corpus.

In [ ]:
def np_link(i): return f"{NP_BASE}/{int(i)}"

def top_features(concept, k=10, lang=None, baseline="neutral"):
    sel = (df.concept == concept) & ((df.lang == lang) if lang else True)
    base = (df.concept == baseline) & ((df.lang == lang) if lang else True)
    spec = F[sel].mean(0) - F[base].mean(0)
    idx = np.argsort(-spec)[:k]
    return pd.DataFrame({"feature": idx,
                         "specificity": spec[idx].round(2),
                         "hit_rate": (F[sel][:, idx] > 0).mean(0).round(2),
                         "neutral_hit_rate": (F[base][:, idx] > 0).mean(0).round(2),
                         "neuronpedia": [np_link(i) for i in idx]})

for c in ["sadness", "grief", "anxiety", "fear"]:
    print(f"\n=== {c.upper()} ===")
    display(top_features(c, k=8))

## 9 · Contrasts: what separates grief from sadness, and fear from anxiety?
Features at the top lean toward A; features at the bottom lean toward B. **Shared** features fire for both. Those are where the model may be blurring the two concepts together.

In [ ]:
def contrast(a, b, k=8, lang=None):
    sa = (df.concept == a) & ((df.lang == lang) if lang else True)
    sb = (df.concept == b) & ((df.lang == lang) if lang else True)
    ma, mb = F[sa].mean(0), F[sb].mean(0)
    d = ma - mb
    lean_a, lean_b = np.argsort(-d)[:k], np.argsort(d)[:k]
    shared = np.argsort(-np.minimum(ma, mb))[:k]
    mk = lambda idx, lab: pd.DataFrame({"feature": idx, lab: True, f"mean_{a}": ma[idx].round(2),
                                        f"mean_{b}": mb[idx].round(2), "neuronpedia": [np_link(i) for i in idx]})
    return mk(lean_a, f"leans_{a}"), mk(lean_b, f"leans_{b}"), mk(shared, "shared")

for a, b in [("grief", "sadness"), ("fear", "anxiety")]:
    la, lb, sh = contrast(a, b)
    print(f"\n##### {a} vs {b}"); print(f"-- leans {a}"); display(la); print(f"-- leans {b}"); display(lb); print("-- shared"); display(sh)

## 10 · Separation scores: the numbers you'll track before and after any change
- **Centroid similarity:** how alike the average feature patterns of two concepts are (1.0 means identical). Lower means the model keeps them more distinct.
- **Probe accuracy:** a simple classifier tries to tell concepts apart using only the features.
- **Cross-lingual transfer:** train the classifier on English, test it on Spanish. High transfer means the model uses a shared, language-independent concept.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning); warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

CONCEPTS = ["sadness", "grief", "anxiety", "fear"]

def centroid_sims(Fm, lang=None):
    cs = {}
    for c in CONCEPTS:
        sel = (df.concept == c) & ((df.lang == lang) if lang else True)
        v = Fm[sel].mean(0); cs[c] = v / (np.linalg.norm(v) + 1e-9)
    return pd.DataFrame([[float(cs[a] @ cs[b]) for b in CONCEPTS] for a in CONCEPTS], index=CONCEPTS, columns=CONCEPTS).round(3)

def probe_scores(Fm):
    emo = df.concept.isin(CONCEPTS).values
    X, y, lang = np.log1p(Fm[emo]), df.concept[emo].values, df.lang[emo].values
    X = X[:, X.max(0) > 0]                      # drop features that never fire on this set
    # pick the 128 most informative features *inside* each training fold, then a plain logistic classifier
    clf = lambda: make_pipeline(SelectKBest(f_classif, k=min(128, X.shape[1])), LogisticRegression(max_iter=2000, C=1.0))
    out = {}
    for L in ["en", "es"]:
        m = lang == L
        out[f"cv_acc_{L}"] = cross_val_score(clf(), X[m], y[m], cv=5).mean()
    for tr, te in [("en", "es"), ("es", "en")]:
        c = clf().fit(X[lang == tr], y[lang == tr])
        out[f"transfer_{tr}->{te}"] = c.score(X[lang == te], y[lang == te])
    return pd.Series(out).round(3)

def pair_sims(Fm):
    rows = {}
    for L in ["en", "es"]:
        s = centroid_sims(Fm, L)
        rows[L] = {"grief~sadness": s.loc["grief", "sadness"], "fear~anxiety": s.loc["fear", "anxiety"]}
    return pd.DataFrame(rows)

print("Centroid similarity (all languages):"); display(centroid_sims(F))
print("Key pairs by language:"); display(pair_sims(F))
print("Probe accuracy (chance = 0.25):"); display(probe_scores(F))

BASELINE = {"pairs": pair_sims(F), "probe": probe_scores(F)}

## 11 · Zoom in on one feature
Pick a feature index from the tables above and see which **words** it fires on.

In [ ]:
FEAT = int(top_features("grief", k=1).feature[0])
print("feature", FEAT, "→", np_link(FEAT), "\n")
token_view("Desde que murió mi papá, sigo poniendo su plato en la mesa.", FEAT)
print()
token_view("Since my father died, I keep setting his plate at dinner.", FEAT)

## 12 · Steering: turn one feature up (or down) while the model writes
This adds the feature's direction to layer 13 on every token. Strength is a fraction of the typical activation size at that layer. Negative values push *away* from the feature. Steer too hard and the output turns to gibberish; finding that edge is part of the experiment.

In [ ]:
with torch.no_grad():
    _h, _m = layer_acts(df.text.tolist()[:32])
    RESID_NORM = float(_h[content_mask(_m).bool()].norm(dim=-1).median())
print("typical residual norm at layer", LAYER, "≈", round(RESID_NORM, 1))

def steer_generate(prompt, feat, strength, m=None, max_new_tokens=70):
    m = m or model
    d = sae.W_dec[feat].detach().float(); d = d / d.norm()
    add = strength * RESID_NORM * d
    def hook(mod, inp, out):
        if isinstance(out, tuple): return (out[0] + add.to(out[0].dtype),) + tuple(out[1:])
        return out + add.to(out.dtype)
    h = decoder_layers(m)[LAYER].register_forward_hook(hook)
    try:
        ids = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                      tokenize=True, return_dict=True, return_tensors="pt")["input_ids"].to(device)
        with torch.no_grad():
            out = m.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    finally:
        h.remove()
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

PROMPT = "Write two sentences about how your week went."
for s in [0.0, 0.15, 0.3, 0.6, -0.3]:
    print(f"--- strength {s:+.2f}\n{steer_generate(PROMPT, FEAT, s)}\n")

## 13 · *(Optional)* LoRA fine-tune, then re-measure
This is the **fix → verify** loop. A small LoRA adapter is trained on example conversations, then sections 5 and 10 run again on the tuned model, so you can see:
- whether grief and sadness (or fear and anxiety) **separated more**,
- whether **cross-lingual transfer** held up,
- how far the SAE map **drifted** (variance explained).

⚠️ The `TRAIN` examples below are **placeholders** that only show the format. Replace them with a reviewed dataset of at least a few hundred examples written or checked by clinicians and native speakers. Training on six examples won't teach the model anything real.

In [ ]:
TRAIN = [
  ("My grandmother died last month and I still set out two cups of coffee.",
   "That sounds like grief, the kind that shows up in small daily habits. Those two cups say a lot about how present she still is for you."),
  ("I'm just kind of down today, nothing big happened.",
   "Some days just feel low without a clear reason. That's sadness, and it often passes. Is anything helping a little today?"),
  ("Se murió mi tío y no he podido llorar todavía.",
   "El duelo no siempre llega con lágrimas enseguida; a veces aparece después. No hay una manera correcta de vivirlo."),
  ("Ando bajito hoy, pero no sé por qué.",
   "Hay días que uno amanece triste sin una razón clara. Es normal. ¿Hay algo pequeño que te haga sentir un poco mejor?"),
  ("I keep worrying something will go wrong at work, even when everything is fine.",
   "That ongoing what-if worry, with no actual threat in front of you, sounds like anxiety rather than fear."),
  ("Un carro casi me atropella y todavía me tiemblan las manos.",
   "Eso fue miedo de verdad, una reacción a un peligro real. Es normal que el cuerpo tarde un rato en calmarse."),
]

from peft import LoraConfig, get_peft_model
lcfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
ft = get_peft_model(model, lcfg)      # note: adds adapters onto `model` in place; baseline numbers were saved above
ft.print_trainable_parameters()

def encode_pair(u, a):
    ids = tok.apply_chat_template([{"role": "user", "content": u}, {"role": "assistant", "content": a}],
                                tokenize=True, return_dict=True, return_tensors="pt")["input_ids"][0]
    return ids

opt = torch.optim.AdamW([p for p in ft.parameters() if p.requires_grad], lr=2e-4)
ft.train()
EPOCHS = 3
for ep in range(EPOCHS):
    tot = 0
    for u, a in TRAIN:
        ids = encode_pair(u, a).unsqueeze(0).to(device)
        loss = ft(input_ids=ids, labels=ids).loss
        loss.backward(); opt.step(); opt.zero_grad(); tot += loss.item()
    print(f"epoch {ep+1}: loss {tot/len(TRAIN):.3f}")
ft.eval()

In [ ]:
print("SAE fit after tuning (drift check):", round(sae_fit(["The library opens at nine on Saturdays.",
                                                            "Desde que murió mi papá, sigo poniendo su plato en la mesa."], ft), 3))
F_ft = feature_matrix(df.text.tolist(), ft)

cmp_pairs = pd.concat({"before": BASELINE["pairs"], "after": pair_sims(F_ft)}, axis=1)
cmp_probe = pd.DataFrame({"before": BASELINE["probe"], "after": probe_scores(F_ft)})
print("Centroid similarity (lower = more distinct):"); display(cmp_pairs)
print("Probe accuracy / transfer:"); display(cmp_probe)
print("\nSame prompt, tuned model:\n", steer_generate(PROMPT, FEAT, 0.0, ft))

## 14 · Save results

In [ ]:
out = df.copy()
for c in CONCEPTS:
    for i, f in enumerate(top_features(c, k=3).feature):
        out[f"{c}_top{i+1}_f{f}"] = F[:, f]
out.to_csv("emotion_map_results.csv", index=False)
pd.concat([top_features(c, k=20).assign(concept=c) for c in CONCEPTS]).to_csv("top_features_by_concept.csv", index=False)
print("Saved emotion_map_results.csv and top_features_by_concept.csv (Files panel, left).")

## Where to go next
- **Bigger map:** switch `SAE_ID` to `layer_13_width_65k_l0_medium` for finer-grained features.
- **Bigger model:** `gemma-3-4b-it` with `SAE_RELEASE = "gemma-scope-2-4b-it-res"`, `SAE_ID = "layer_17_width_16k_l0_medium"`, `LAYER = 17`, `NP_BASE = "https://www.neuronpedia.org/gemma-3-4b-it/17-gemmascope-2-res-16k"`. This needs float16/bfloat16, or a bigger GPU than a T4.
- **More concepts:** literal vs. culturally appropriate translation, *nervios*, *ataque de nervios*, emotional intensity vs. expression.
- **Better data:** expand the probe set to 100+ sentences per concept per language, with native-speaker and clinical review.